In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
pd.options.display.max_colwidth = 1000

In [ ]:
#read in All Jobs report
df_Jobs = pd.read_csv(r'')
df_Jobs

In [ ]:
#filter all jobs logged between october 2024 to March 2025 
# Ensure 'DateLogged' is in datetime format with specific format
df_Jobs['DateLogged'] = pd.to_datetime(df_Jobs['DateLogged'], format='%d/%m/%Y %H:%M', errors='coerce')

# Define the date range
start_date = "01-10-2024"
end_date = "31-03-2025"


# Filter the dataframe
df_Jobs = df_Jobs[(df_Jobs['DateLogged'] >= start_date) & (df_Jobs['DateLogged'] <= end_date)]

# Display the result
df_Jobs

In [ ]:
#Clean the All Jobs
#AllJobs
df_AllJobs = df_Jobs[['Job Number','Job Description','DateLogged','TargetAttendanceDate','CompletedDate','Job Type','Job Category','Job Trade','Status','Priority','Customer','Site','Site Postcode','Area','Job Tags','First Engineer']]
df_AllJobs = df_AllJobs.applymap(lambda x: x.upper() if isinstance(x, str) else x)
df_AllJobs = df_AllJobs.loc[df_AllJobs['Status'] == 'COMPLETED']
df_AllJobs = df_AllJobs.dropna(subset=['Site Postcode'])
df_AllJobs['DateLogged'] = pd.to_datetime(df_AllJobs['DateLogged'], format='%d/%m/%Y %H:%M', errors='coerce')
df_AllJobs['TargetAttendanceDate'] = pd.to_datetime(df_AllJobs['TargetAttendanceDate'], format='%d/%m/%Y %H:%M', errors='coerce')
df_AllJobs['CompletedDate'] = pd.to_datetime(df_AllJobs['CompletedDate'], format='%d/%m/%Y %H:%M', errors='coerce')
df_AllJobs = df_AllJobs.dropna(subset=['DateLogged', 'TargetAttendanceDate', 'CompletedDate'])


df_AllJobs = df_AllJobs[~df_AllJobs['Job Tags'].str.contains('COP|MBMS', case=False, na=False)]
df_AllJobs = df_AllJobs[~df_AllJobs['Job Type'].str.contains('PPM|POSTAL|COMPLAINT|MBMS|AUDIT|CODE OF PRACTICE', case=False, na=False)]
df_AllJobs = df_AllJobs[~df_AllJobs['Priority'].str.contains('PPM', case=False, na=False)]
df_AllJobs['Priority'] = df_AllJobs['Priority'].str.replace(r'^\D*\s*', '', regex=True)
df_AllJobs = df_AllJobs[df_AllJobs['Priority'] != '']
df_AllJobs = df_AllJobs.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df_AllJobs['Job Trade'] = df_AllJobs['Job Trade'].fillna('BBG')


In [ ]:
df_AllJobs.rename({"Site Postcode":"Postcode"},axis=1,inplace=True)
df_AllJobs

In [ ]:
#Read in Uk Postcodes to get the Longitude and Latitude for all Job sites
df_Ukpostcodes = pd.read_csv(r'')
df_Ukpostcodes

In [ ]:
#Merge df_AllJobs to df_UKpostcodes on Postcode
df_AllJobs = df_AllJobs.merge(df_Ukpostcodes, on ="Postcode", how ='left')

In [ ]:
# Strip whitespace and extract first half of the postcode as Area district
df_AllJobs['Postcode'] = df_AllJobs['Postcode'].str.strip()  # Remove leading/trailing spaces
df_AllJobs['District'] = df_AllJobs['Postcode'].str.extract(r'(\S+)') #extract first half of the postcode
df_AllJobs

In [ ]:
#read in internal Zone classification
df_Areas = pd.read_excel(r'', sheet_name="")
df_Areas
df_Areas = df_Areas[['Postcode','Zone Name']]

In [ ]:
df_Areas.rename({"Postcode":"District"},axis=1,inplace=True)

In [ ]:
#Merge df_Areas to df_AllJobs
df_AllJobs = df_AllJobs.merge(df_Areas, on ="District", how ='left')
df_AllJobs

In [ ]:
#Compare the complete date to the Target attendance date to see Jobs that were completed within Service level agreement
df_AllJobs['InTime'] = (df_AllJobs['TargetAttendanceDate'] >= df_AllJobs['CompletedDate']) | df_AllJobs['TargetAttendanceDate'].isna()
df_AllJobs

In [ ]:
#Count total Jobs, sum the amount of Jobs completed in time, take the mean of longitude and latitude - Grouping all Jobs by district 
sla_performance = df_AllJobs.groupby(['District']).agg(
    total_jobs=pd.NamedAgg(column='Job Number', aggfunc='count'),
    jobs_in_time=pd.NamedAgg(column='InTime', aggfunc='sum'),
    Latitude=pd.NamedAgg(column='uk latitude', aggfunc='mean'),
    Longitude=pd.NamedAgg(column='uk longitude', aggfunc='mean'),
    Zone_name=pd.NamedAgg(column='Zone Name', aggfunc=lambda x: ', '.join(x.dropna().map(str).unique())),
    #Job_Type_Weight=pd.NamedAgg(
        #column='Job Type', 
        #aggfunc=lambda x: np.where(x == 'BREAKDOWN', 2, np.where(x == 'INSTALLATION', 1, 0)).sum())
).reset_index()


In [ ]:
# Calculate SLA percentage and handle division by zero
sla_performance['sla_percentage'] = np.where(
    sla_performance['total_jobs'] == 0, 
    100, 
    (sla_performance['jobs_in_time'] / sla_performance['total_jobs']) * 100
)


sla_performance.dropna(inplace=True)
sla_performance

In [ ]:
#Identify the regions with the Highest Jobs but Lowest SLA Pass percentage

# Filter districts with SLA below 80%
low_sla_df = sla_performance[sla_performance['sla_percentage'] < 80]

# Sort by total_jobs to find where the workload is high despite poor SLA
result_districts = low_sla_df.sort_values(by='total_jobs', ascending=False)

result_districts

In [ ]:
#write to excel
result_districts.to_excel('LowSLApass.xlsx')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium

# Create map centered on average coordinates
center_lat = sla_performance['Latitude'].mean()
center_lon = sla_performance['Longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=7, tiles='CartoDB positron')

# Add job density circles (blue, size based on total_jobs)
for _, row in sla_performance.iterrows():
    folium.CircleMarker(
        location=(row['Latitude'], row['Longitude']),
        radius=row['total_jobs'] * 0.2,
        color='blue',  # Blue for job count circles
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        popup=(f"<b>District:</b> {row['District']}<br>"
               f"<b>Total Jobs:</b> {row['total_jobs']}<br>"
               f"<b>SLA %:</b> {row['sla_percentage']:.1f}%")
    ).add_to(m)

# Add SLA percentage circles (colored by SLA %)
for _, row in sla_performance.iterrows():
    sla_percentage = row['sla_percentage']

    # Define color based on SLA percentage ranges
    if sla_percentage >= 90:
        color = 'green'  # High SLA: Green
    elif sla_percentage >= 70:
        color = 'yellow'  # Mid SLA: Purple
    else:
        color = 'red'  # Low SLA: Red
    
    folium.CircleMarker(
        location=(row['Latitude'], row['Longitude']),
        radius=6,
        color=color,  # Different color based on SLA percentage
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=(f"<b>District:</b> {row['District']}<br>"
               f"<b>SLA %:</b> {sla_percentage:.1f}%")
    ).add_to(m)

# Add custom legend
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 220px;
    height: 180px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
<b>Map Legend</b><br>
<span style="color:blue;">●</span> <b>Blue Circle Size</b>: Job Count<br>
<span style="color:red;">●</span> <b>Red</b>: Low SLA % (below 70%)<br>
<span style="color:yellow;">●</span> <b>yellow</b>: Mid SLA % (70% - 89%)<br>
<span style="color:green;">●</span> <b>Green</b>: High SLA % (90% and above)<br><br>
<b>Note:</b> Color scale = SLA %
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Display map
m



In [ ]:
from folium.plugins import HeatMap

# Get min/max job values for scale labels
min_jobs = sla_performance['total_jobs'].min()
max_jobs = sla_performance['total_jobs'].max()
 
# Create base map
center_lat = sla_performance['Latitude'].mean()
center_lon = sla_performance['Longitude'].mean()
heat_map = folium.Map(location=[center_lat, center_lon], zoom_start=7, tiles='CartoDB positron')
 
# Prepare heatmap data
heat_data = sla_performance[['Latitude', 'Longitude', 'total_jobs']].values.tolist()
 
# Add heat layer
HeatMap(
    data=heat_data,
    radius=5,
    blur=2,
    max_zoom=15
).add_to(heat_map)
 
# Add legend with actual job counts
legend_html = f"""
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 220px;
    height: 95px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
<b>Job Density (Total Jobs)</b><br>
<div style="height: 15px;
     background: linear-gradient(to right, blue, lime, yellow, orange, red);
     margin-top: 5px;
     margin-bottom: 5px;"></div>
<span style="float:left;">{min_jobs} Jobs</span>
<span style="float:right;">{max_jobs} Jobs</span><br style="clear:both;">
</div>
"""
 
heat_map.get_root().html.add_child(folium.Element(legend_html))
 
# Display map
heat_map

In [ ]:
# --- Filter districts with SLA % < 80 ---
filtered_sla = sla_performance[sla_performance['sla_percentage'] < 80].copy()
 
# --- Normalize SLA % for color (0% = red, 100% = green) ---
norm = plt.Normalize(0, 100)
cmap = plt.cm.RdYlGn
filtered_sla['color'] = filtered_sla['sla_percentage'].apply(lambda x: mcolors.to_hex(cmap(norm(x))))
 
# --- Create folium map centered on the data ---
center_lat = filtered_sla['Latitude'].mean()
center_lon = filtered_sla['Longitude'].mean()
enhanced_map = folium.Map(location=[center_lat, center_lon], zoom_start=7, tiles='CartoDB positron')
 
# --- Add markers (sorted so lowest SLA is drawn on top) ---
for _, row in filtered_sla.sort_values(by='sla_percentage', ascending=False).iterrows():
    radius = max(5, row['total_jobs'] * 0.2)
    popup_html = (
        f"<b>District:</b> {row['District']}<br>"
        f"<b>SLA %:</b> {row['sla_percentage']:.1f}%<br>"
        f"<b>Total Jobs:</b> {row['total_jobs']}"
    )
    folium.CircleMarker(
        location=(row['Latitude'], row['Longitude']),
        radius=radius,
        color=row['color'],
        fill=True,
        fill_color=row['color'],
        fill_opacity=0.85,
        popup=popup_html
    ).add_to(enhanced_map)
 
# --- Add custom legend ---
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 240px;
    height: 105px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
<b>SLA % (Filtered &lt; 80%)</b><br>
<div style="height: 15px;
     background: linear-gradient(to right, red, yellow, green);
     margin-top: 5px;
     margin-bottom: 5px;"></div>
<span style="float:left;">0%</span>
<span style="float:right;">100%</span><br style="clear:both;">
<br><b>Circle size</b> = Job Count
</div>
"""
enhanced_map.get_root().html.add_child(folium.Element(legend_html))
 
# --- Display map 
enhanced_map
 